In [10]:
import pandas as pd
from openai import OpenAI
from autoddg import AutoDDG
from autoddg.utils import get_sample
from autoddg.evaluation import BaseEvaluator
from typing import Optional
from evaluate import load
# --- Import custom files ---
from prompts import ALL_RELATED_WORK_PROMPTS
from utils import log_result, run_description_experiment
import os 
import json
from cache_utils import run_with_caching, load_profile_from_cache#, MockAutoDDG           #Mock is for testing



In [11]:
# --- LLM Config ---
MODEL_CONFIG = {
    "base_url": "http://localhost:11434/v1",
    "api_key": "ollama",
    "model_name": "llama3.1:8b",
}

# Initialize Core Tools

client = OpenAI(api_key=MODEL_CONFIG["api_key"], base_url=MODEL_CONFIG["base_url"])


In [12]:
import json
import pandas as pd

results = pd.read_csv("results-70b.csv")
DATABASE_PATH_ = "../src/autoddg/database.json"

with open(DATABASE_PATH_, "r", encoding="utf-8") as f:
    db = json.load(f)

# dataset_name -> description lookup
name_to_desc = {
    v["dataset_name"].strip(): v.get("description", "")
    for v in db.values()
    if "dataset_name" in v
}

# add column (no overwriting other columns)
results["Reference_Description"] = (
    results["Dataset_Name"].astype(str).str.strip().map(name_to_desc)
)

# optional: keep blanks instead of NaN
results["Reference_Description"] = results["Reference_Description"].fillna("")

results.to_csv("results_refdesc.csv", index=False)

In [ ]:
import pandas as pd
from enhanced_eval import evaluate_all

# Step 1: load results file
df = pd.read_csv("results_refdesc.csv")

# Step 2: compute metrics for each row ( generation)

output=[]

for _, row in df.iterrows():
    metrics = evaluate_all(
        row= row,
        client= client, 
        model_name= MODEL_CONFIG["model_name"] )
    
    # merge original row data + metrics into one dict
    row_and_metrics = {**row.to_dict(), **metrics}
    output.append(row_and_metrics)
    

# Step 3: convert to DataFrame
metrics_df = pd.DataFrame(output)

# Step 4: save full evaluation metrics
metrics_df.to_csv("results-70b-eval.csv", index=False)

print("Enhanced evaluation saved to results-70b-eval.csv")

NameError: name 'i' is not defined

In [16]:
import pandas as pd, json
from enhanced_eval import extract_dataset_profile

df = pd.read_csv("results_refdesc.csv")
row = df.iloc[0]  # pick whichever

profile = extract_dataset_profile(row["Description_Text"], client, MODEL_CONFIG["model_name"])
print(json.dumps(profile, indent=2))

{
  "basic_info": {
    "dataset_name": null,
    "domain_or_field": "immune cell analysis or cytokine profiling",
    "primary_purpose": null
  },
  "data_characteristics": {
    "size_or_scale": null,
    "data_format": null,
    "data_types": "demographic information, measurements of immune cell subsets, cytokine levels",
    "temporal_coverage": null,
    "sample_unit": null
  },
  "provenance": {
    "collection_method": null,
    "data_source": null,
    "collection_date": null,
    "creators_or_curators": null,
    "preprocessing_steps": null
  },
  "usage_context": {
    "typical_applications": "analyze immune responses, identify patterns in cytokine profiles",
    "research_questions_addressed": "investigate the effects of vaccines and infections on the immune system",
    "how_used_in_paper": null,
    "benchmark_or_evaluation_role": null
  },
  "quality_and_limitations": {
    "known_limitations": null,
    "biases_or_caveats": null,
    "quality_issues": null,
    "challeng